# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saif-Eldeen-95/flyrank_internship_ml_saif/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring (Lane 2).**

FlyRank's editors cannot manually review every piece of content a client owns. Across the
32 clients in the starter dataset, 43.8% of pages (13,152 of 30,000) are currently
`trend_direction == "down"` *and* still pulling real search demand (`impressions_90d >= 100`).
That is far more candidates than any editorial team can look at in a week, which is exactly the
kind of "too many, too messy to rank by hand" situation a scoring/ranking model is built for.

I am choosing this lane over the others because:
- the starter repo already ships a working reference pipeline for it (`scripts/01-05`), so I can
  validate my own reasoning against a known-good baseline before I extend it;
- the output (a ranked queue with reason codes) maps directly onto a real decision an editor
  makes every week, unlike clustering (Lane 3) or pure signal analysis (Lane 1), which describe
  the landscape but do not by themselves tell someone what to do next;
- it lets me keep Lane 4 (CTR/engagement scoring) as a sub-question inside this lane later,
  since CTR-weak pages are one of the reason codes a refresh queue already tracks.

This is a **provisional** choice — I can confirm or change it through the end of Week 4.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("rows:", len(df))
print("clients:", df["client_id"].nunique())


rows: 30000
clients: 32


## 2. The question: decision, action, cost of a wrong call

**Research question:** Among a client's currently-visible content pages, which pages should an
editor review *first* for refresh, expansion, protection, or pruning?

**Unit of analysis:** one content page (`content_id`) per client (`client_id`), summarized over
its trailing 90-day window.

**Output:** a ranked queue of pages per client, each with a priority score and a short
human-readable reason code (e.g. `declining_with_demand`, `low_ctr_visible_page`).

**Decision it improves:** which of hundreds of "possibly worth reviewing" pages an editor spends
their limited weekly review capacity on, instead of picking pages by gut feel or working through
them in whatever order they happen to appear.

**Who acts, and what they do:** a content editor / SEO reviewer with a fixed weekly review
budget (the starter pipeline's own queue, for example, has 3,605 "high-confidence" items out of
30,000 rows — still far more than one person reviews in a week). They open the top of the queue,
read the reason codes, and decide whether to refresh the copy, expand it, leave it alone
(monitor), or in rare cases retire it.

**Cost of a wrong call:**
- *False positive* (page ranked high but wasn't actually a priority): editor time is spent on a
  page that didn't need it — a real but bounded cost, since editor hours are the scarce resource.
- *False negative* (a genuinely declining, high-demand page ranked low or missed): the page keeps
  losing visibility/clicks unnoticed, which is a larger and less visible cost because nobody was
  looking for it.
- Because missing a real decline is more expensive than reviewing an unnecessary page, a ranking
  that favors recall among the highest-demand pages is worth more here than one that only
  optimizes overall accuracy.

**Why data or ML can help at all:** a single hand-written rule struggles here because the signal
is spread across many correlated columns (position, CTR, engagement, age, freshness, word count)
that interact — a page can be "fine" on any one signal and still be a priority once several
weak signals stack up. Section 3 shows a plain rule (`stale_visible_page`) catches almost nothing
on its own, while a broader, learned combination of signals does far better — that gap is the
argument for going past a single if-statement.

## 3. Quick look at the data (2-3 real numbers)

*(computed in the cell below, on `data/raw/content_refresh_anonymized.csv`)*

In [2]:
total = len(df)

declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()
low_ctr_visible = ((df["impressions_90d"] >= 500) & (df["avg_position"] > 0)
                    & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)).sum()
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()

print(f"declining_with_demand: {declining_with_demand:,} rows ({declining_with_demand/total:.1%} of {total:,})")
print(f"low_ctr_visible_page:  {low_ctr_visible:,} rows ({low_ctr_visible/total:.1%})")
print(f"stale_visible_page (plain rule only): {stale_visible:,} rows ({stale_visible/total:.1%})")


declining_with_demand: 13,152 rows (43.8% of 30,000)
low_ctr_visible_page:  9,759 rows (32.5%)
stale_visible_page (plain rule only): 17 rows (0.1%)


## 4. Careful words: what I can and can't claim

**I can claim:**
- *observed* rates and counts directly from the starter dataset (e.g. "43.8% of pages are
  currently flagged declining-with-demand in this 30,000-row anonymized slice");
- *directional* patterns — which signals tend to move together, and by roughly how much;
- *decision-support* output — a ranked list with reason codes that helps a human reviewer spend
  limited time more efficiently than picking pages at random or in an arbitrary order.

**I cannot claim:**
- that refreshing a page *causes* it to recover — that needs a controlled experiment or a proper
  causal design, not an observational ranking;
- that I am predicting Google's ranking algorithm, or any of its specific factors;
- that the current label (`trend_direction == "down"`, a bucket computed from the *current*
  window) is a clean target — it is a proxy for "already declining right now," not a future
  outcome. A stronger version of this project, later in the internship, would move to a
  forward-looking label (features from the trailing 90 days predicting decline/recovery over
  the *next* 30 days) so the model is judged on what it foresees, not what it already sees.
- anything about a specific client, domain, URL, or query — the data is pseudonymized and stays
  that way in every output.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.